# Medallion Architecture

In [0]:
#Create database 
spark.sql("""
CREATE DATABASE IF NOT EXISTS ecommerce_ma
""")


DataFrame[]

In [0]:
df_raw = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",
    header=True,
    inferSchema=True
)


In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

bronze_df = (
    df_raw
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", input_file_name())
)


In [0]:
vol = spark.sql("SHOW VOLUMES IN workspace.ecommerce")
display(vol)

database,volume_name
ecommerce,ecommerce_data


In [0]:
from pyspark.sql.functions import col

bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")
    .withColumn("source_file", col("_metadata.file_path"))
)



In [0]:
bronze_path = "/Volumes/workspace/ecommerce/ecommerce_data/bronze/events"

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)


In [0]:
silver_source = spark.read.format("delta").load(
    "/Volumes/workspace/ecommerce/ecommerce_data/bronze/events"
)


In [0]:

from pyspark.sql.functions import col

silver_df = (
    silver_source
    .filter(col("event_type").isin("view", "cart", "purchase"))
    .filter(col("price").isNotNull())
    .filter(col("category_code").isNotNull())
    .dropDuplicates(["user_id", "product_id", "event_time"])
)


In [0]:
silver_path = "/Volumes/workspace/ecommerce/ecommerce_data/silver/events"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)


In [0]:
#read from silver 
silver_events = spark.read.format("delta").load(
    "/Volumes/workspace/ecommerce/ecommerce_data/silver/events"
)


In [0]:
#revenue by category
from pyspark.sql.functions import sum

gold_revenue_by_category = (
    silver_events
    .filter(col("event_type") == "purchase")
    .groupBy("category_code")
    .agg(sum("price").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)
display(gold_revenue_by_category)

category_code,total_revenue
electronics.smartphone,1.7782131698000118E8
electronics.video.tv,1.2457004279999996E7
computers.notebook,1.0678429709999999E7
electronics.clocks,6552737.24999998
appliances.kitchen.washer,5801822.699999988
electronics.audio.headphone,5669469.279999987
appliances.kitchen.refrigerators,4722657.300000001
appliances.environment.vacuum,2762311.62
computers.desktop,1556586.239999999
electronics.tablet,1520144.5999999996


In [0]:
gold_path = "/Volumes/workspace/ecommerce/ecommerce_data/gold/revenue_by_category"

gold_revenue_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)
